In [1]:
import ee
import geemap
from dotenv import load_dotenv
import os
import ipywidgets as widgets
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd

load_dotenv(dotenv_path='.env')
project_name = os.getenv("project_name")
# print(f"Project Name: {project_name}")
ee.Initialize(project=project_name)

In [6]:
#according to 2022 census, 
district_data_raw = {
  'Barguna': 1035596, 'Barisal': 2634203, 'Bhola': 1980452, 'Jhalokati': 677559, 
  'Patuakhali': 1770096, 'Pirojpur': 1227915, 'Bandarban': 495252, 'Brahamanbaria': 3403786, 
  'Chandpur': 2713247, 'Chittagong': 9439076, "Cox's Bazar": 2906281, 'Comilla': 6394875, 
  'Feni': 1697379, 'Khagrachhari': 735116, 'Lakshmipur': 1994930, 'Noakhali': 3732042, 
  'Rangamati': 666627, 'Dhaka': 15210851, 'Faridpur': 2232772, 'Gazipur': 5433538, 
  'Gopalganj': 1336907, 'Kishoreganj': 3373219, 'Madaripur': 1334811, 'Manikganj': 1608372, 
  'Munshiganj': 1677941, 'Narayanganj': 4035461, 'Narsingdi': 2667968, 'Rajbari': 1228267, 
  'Shariatpur': 1336396, 'Tangail': 4168083, 'Jamalpur': 2583984, 'Mymensingh': 6097814, 
  'Netrakona': 2403205, 'Sherpur': 1552469, 'Bagerhat': 1649874, 'Chuadanga': 1262205, 
  'Jessore': 3146317, 'Jhenaidah': 2051607, 'Khulna': 2673002, 'Kushtia': 2198731, 
  'Magura': 1056683, 'Meherpur': 721447, 'Narail': 806662, 'Satkhira': 2246691, 
  'Bogra': 3815192, 'Nawabganj': 1875290, 'Joypurhat': 977150, 'Naogaon': 2844921, 
  'Natore': 1900213, 'Pabna': 2972654, 'Rajshahi': 2978156, 'Sirajganj': 3430443, 
  'Dinajpur': 3392251, 'Gaibandha': 2621756, 'Kurigram': 2383268, 'Lalmonirhat': 1461589, 
  'Nilphamari': 2141180, 'Panchagarh': 1207252, 'Rangpur': 3243247, 'Thakurgaon': 1569529, 
  'Habiganj': 2440151, 'Maulvibazar': 2196601, 'Sunamganj': 2788358, 'Sylhet': 3990003
}

district_data = ee.Dictionary(district_data_raw)

In [7]:
bangladesh_districts = ee.FeatureCollection("FAO/GAUL/2015/level2") \
    .filter(ee.Filter.eq('ADM0_NAME', 'Bangladesh'))

In [10]:
def add_population(feature):
    name = feature.get('ADM2_NAME')
    pop = district_data.get(name)
    return feature.set('population_2022', pop)

population_map = bangladesh_districts.map(add_population)

pop_image = population_map.reduceToImage(
    properties=['population_2022'],
    reducer=ee.Reducer.first()
)

vis_params = {
    'min': 500000,
    'max': 5000000,
    'palette': ['e5f5e0', 'a1d99b', '31a354', 'ffeda0', 'feb24c', 'f03b20', 'bd0026']
}

m = geemap.Map()
m.centerObject(bangladesh_districts, 7)

m.addLayer(pop_image, vis_params, 'Population Map 2022')
borders = ee.Image().paint(population_map, 0, 1)
m.addLayer(borders, {'palette': 'black'}, 'District Borders')

m.add_colorbar(vis_params, label="Population", layer_name="Population Map 2022")
print('Sample District Data:')
print(population_map.limit(5).getInfo())
m

Sample District Data:
{'type': 'FeatureCollection', 'columns': {'ADM0_CODE': 'Integer', 'ADM0_NAME': 'String', 'ADM1_CODE': 'Integer', 'ADM1_NAME': 'String', 'ADM2_CODE': 'Integer', 'ADM2_NAME': 'String', 'DISP_AREA': 'String', 'EXP2_YEAR': 'Integer', 'STATUS': 'String', 'STR2_YEAR': 'Integer', 'Shape_Area': 'Float', 'Shape_Leng': 'Float', 'population_2022': 'Object', 'system:index': 'String'}, 'version': 1701682634012831, 'id': 'FAO/GAUL/2015/level2', 'properties': {'system:asset_size': 316969462}, 'features': [{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[90.2350110465334, 22.45421363510117], [90.23531427332811, 22.454244845062533], [90.23550153638247, 22.454449929961143], [90.2356531513968, 22.454623871832958], [90.23587162695195, 22.454846827312522], [90.23609010373465, 22.455078695175448], [90.23621501974682, 22.455212386131738], [90.23633986457483, 22.455314973508926], [90.23646470980366, 22.45537299718604], [90.23660294200113, 22.455435366662954], [90.236

Map(center=[23.82705199221704, 90.28837386642488], controls=(WidgetControl(options=['position', 'transparent_b…